In [ ]:
import pandas as pd
from pytrends.request import TrendReq
import time

all_queries = [
    # Geopolitical / War
    "iran israel war", "strait of hormuz", "middle east war", "world war 3", "iran attack",
    "israel bombing", "us iran war", "hormuz blockade", "oil tanker attack", "middle east conflict",
    # Oil & Energy
    "oil prices", "crude oil price", "brent crude", "petrol price india", "fuel price india",
    "oil supply shortage", "opec oil cut", "energy crisis", "oil price rise", "diesel price india",
    # Indian Market Panic
    "nifty crash", "sensex crash", "nifty fall today", "stock market crash india", "sensex today",
    "nifty today", "should i sell stocks", "stock market india", "nifty prediction", "market circuit breaker",
    # Macro / Economy Fear
    "inflation india", "recession 2026", "dollar rupee", "rupee fall", "india gdp",
    "rbi rate hike", "india economy war", "war inflation", "global recession", "import export india",
    # Sector Specific
    "aviation fuel price", "indigo airlines", "spicejet shares", "ongc share price", "reliance share price",
    "defence stocks india", "hal share price", "shipping stocks india", "gold price india", "gold rate today",
    # Retail Investor Behaviour
    "buy gold now", "safe investment india", "best stocks to buy now", "where to invest during war",
    "mutual fund safe", "fd vs stocks", "sell stocks now", "war proof investment", "dollar buy india",
    "sovereign gold bond",
]

TFRAME = "2026-02-28 2026-04-15"
results = {}  # query -> Series

round_num = 0
while len(results) < len(all_queries):
    round_num += 1
    pending = [q for q in all_queries if q not in results]
    print(f"\n=== Round {round_num} | {len(results)}/{len(all_queries)} done | {len(pending)} pending ===")

    for query in pending:
        print(f"  Fetching: '{query}' ...", end=" ")
        try:
            pytrends = TrendReq(hl='en-IN', tz=330)
            pytrends.build_payload([query], timeframe=TFRAME, geo='')
            df = pytrends.interest_over_time()
            if not df.empty:
                results[query] = df[query].rename(query)
                print(f"OK")
            else:
                print(f"empty, will retry")
        except Exception as e:
            print(f"error ({e}), will retry")
        time.sleep(20)

print(f"\nAll {len(all_queries)} queries fetched. Combining...")

trends_df = pd.concat(results.values(), axis=1)
trends_df.index.name = "date"
trends_df.index = pd.to_datetime(trends_df.index).date
trends_df = trends_df[all_queries]  # restore original column order

print(f"Shape: {trends_df.shape}")
print(f"Dates: {trends_df.index.min()} → {trends_df.index.max()}")
print(f"\nSample:\n{trends_df.head()}")

trends_df.to_csv("google_trends_raw.csv")
print("\nSaved: google_trends_raw.csv")

Total queries: 60
Total batches: 12
  Fetching batch 1/12: ['iran israel war', 'strait of hormuz', 'middle east war', 'world war 3', 'iran attack']


KeyboardInterrupt: 